# Covariance and Log-Euclidean SPD Conditioning

This notebook tests whether explicitly representing the recent covariance state improves the retained **Neural CDE-conditioned diffusion** model, and whether a manifold-aware Log-Euclidean representation adds value beyond an ordinary covariance control.

For each 60-day raw excess-return history \(X\in\mathbb{R}^{60\times12}\), Phase 2A constructs a regularized sample covariance matrix

\[
\Sigma=\frac{1}{T-1}(X-\bar X)^\top(X-\bar X),
\]

then compares two matched 78-dimensional symmetric-matrix representations:

1. **Covariance control:** \(\operatorname{svec}(\Sigma)\)
2. **Log-SPD:** \(\operatorname{svec}(\log\Sigma)\)

Both are added to the frozen Phase-0 CDE condition through the same zero-initialized residual projection. The key scientific question is not merely whether covariance helps, but whether **Log-SPD improves over ordinary covariance**.

All feature normalization is fitted on the training period only. The held-out test split is not evaluated.


## Setup


In [1]:
import os
import subprocess

REPO_PATH = "/kaggle/working/diffusion-portfolio"

REPO_URL = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)

if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_PATH,
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "fetch",
        "origin",
        "main",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

CURRENT_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        REPO_PATH,
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

os.chdir(
    REPO_PATH
)

print(
    "Synced to latest main:"
)
print(
    CURRENT_COMMIT
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)


Cloning into '/kaggle/working/diffusion-portfolio'...


HEAD is now at 8e7f9ee Add Phase 2 SPD ablation pipeline
Synced to latest main:
8e7f9ee053c38cb68a5e5ee1b7d62b352846b188
8e7f9ee Add Phase 2 SPD ablation pipeline


From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [2]:
!pip install -q -e '.[research]'

import sys

SRC_PATH = f"{REPO_PATH}/src"

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH,
    )


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 43.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.6 MB/s eta 0:00:00
  Building editable for diffusion-portfolio (pyproject.toml) ... done


### Tests

Run the relevant feature, conditioning, diffusion, and training regression tests before executing Phase 2A.


In [3]:
!pytest -q \
    tests/unit/test_spd_features.py \
    tests/unit/test_precomputed_conditioning.py \
    tests/unit/test_conditional_diffusion_model.py \
    tests/unit/test_diffusion_training.py


................................                                         [100%]


### Restore

Restore the retained Phase-0 CDE artifact and any available Phase-2A outputs. Missing Phase-2A backups are recomputed or resumed.


In [4]:
from pathlib import Path
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

SOURCE = "phase0/neural_cde_rk4x4"

DESTINATION = Path(
    "/kaggle/working/phase0_cde"
)

required = [
    "config.yaml",
    "standardizer.npz",
    "best.pt",
]

if DESTINATION.exists():
    print(
        "Already available:",
        DESTINATION,
    )

else:
    DESTINATION.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filename in required:
        downloaded = Path(
            kagglehub.dataset_download(
                HANDLE,
                path=(
                    f"{SOURCE}/{filename}"
                ),
            )
        )

        shutil.copy2(
            downloaded,
            DESTINATION / filename,
        )

        print(
            "Restored:",
            filename,
        )

for filename in required:
    path = (
        DESTINATION
        / filename
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

print(
    "Phase 0 CDE artifact is ready."
)

Restored: config.yaml
Restored: standardizer.npz
Restored: best.pt
Phase 0 CDE artifact is ready.


## 1. SPD feature feasibility

The covariance matrices are computed from **raw historical excess returns**, not standardized CDE inputs. A small relative eigenvalue floor guarantees strict positive definiteness before applying the matrix logarithm.

The symmetric matrices are represented using `svec`, where off-diagonal terms are scaled by \(\sqrt{2}\). This preserves Frobenius geometry under Euclidean vectorization and yields

\[
\frac{12(12+1)}{2}=78
\]

features for both Covariance and Log-SPD.

Before training, we verify feature dimensionality, finite values, train-only standardization, absence of degenerate columns, and the empirical covariance spectrum.


In [5]:
!PYTHONUNBUFFERED=1 python -u scripts/extract_features.py --kind spd \
    --cde-run-dir /kaggle/working/phase0_cde \
    --output-dir /kaggle/working/phase2_features \
    --batch-size 128 \
    --eigenvalue-floor 1e-6 \
    --device cuda


PHASE 2 SPD FEATURE PRECOMPUTATION
Device: cuda
Phase 0 CDE epoch: 42
Eigenvalue floor: 1e-06

Extracting training features...
Extracting validation features...

Train samples: 10516
Validation samples: 1256
CDE condition: 128
Covariance features: 78
Log-SPD features: 78
Covariance constant columns: []
Log-SPD constant columns: []

Train spectrum: {'min_eigenvalue_min': 8.714945920473838e-07, 'min_eigenvalue_median': 2.5683925741759595e-06, 'max_eigenvalue_median': 0.0005576540715992451, 'condition_number_median': 224.7659454345703, 'condition_number_p95': 573.766357421875, 'condition_number_max': 2489.1533203125}
Validation spectrum: {'min_eigenvalue_min': 2.3346690340986243e-06, 'min_eigenvalue_median': 9.305320418206975e-06, 'max_eigenvalue_median': 0.0013601714745163918, 'condition_number_median': 182.4652099609375, 'condition_number_p95': 409.4076232910156, 'condition_number_max': 543.7298583984375}

Artifacts: /kaggle/working/phase2_features
TEST SPLIT WAS NOT TOUCHED.


In [6]:
import json
import numpy as np

train = np.load(
    "/kaggle/working/phase2_features/train.npz"
)

validation = np.load(
    "/kaggle/working/phase2_features/validation.npz"
)

with open(
    "/kaggle/working/phase2_features/metadata.json",
    encoding="utf-8",
) as handle:
    feature_metadata = json.load(
        handle
    )

print(
    "Train CDE:",
    train[
        "cde_condition"
    ].shape,
)

print(
    "Train covariance:",
    train[
        "covariance"
    ].shape,
)

print(
    "Train Log-SPD:",
    train[
        "log_spd"
    ].shape,
)

print(
    "Validation covariance:",
    validation[
        "covariance"
    ].shape,
)

print(
    "Validation Log-SPD:",
    validation[
        "log_spd"
    ].shape,
)

print()
print(
    "Covariance constant indices:",
    feature_metadata[
        "covariance_constant_indices"
    ],
)

print(
    "Log-SPD constant indices:",
    feature_metadata[
        "log_spd_constant_indices"
    ],
)

print()
print(
    "Covariance train max |mean|:",
    float(
        np.abs(
            train[
                "covariance"
            ].mean(
                axis=0
            )
        ).max()
    ),
)

print(
    "Covariance train max |std - 1|:",
    float(
        np.abs(
            train[
                "covariance"
            ].std(
                axis=0
            )
            - 1.0
        ).max()
    ),
)

print(
    "Log-SPD train max |mean|:",
    float(
        np.abs(
            train[
                "log_spd"
            ].mean(
                axis=0
            )
        ).max()
    ),
)

print(
    "Log-SPD train max |std - 1|:",
    float(
        np.abs(
            train[
                "log_spd"
            ].std(
                axis=0
            )
            - 1.0
        ).max()
    ),
)

print()
print(
    "Train spectrum:",
    feature_metadata[
        "train_spectrum"
    ],
)

print(
    "Validation spectrum:",
    feature_metadata[
        "validation_spectrum"
    ],
)


Train CDE: (10516, 128)
Train covariance: (10516, 78)
Train Log-SPD: (10516, 78)
Validation covariance: (1256, 78)
Validation Log-SPD: (1256, 78)

Covariance constant indices: []
Log-SPD constant indices: []

Covariance train max |mean|: 5.074528530712996e-07
Covariance train max |std - 1|: 2.562999725341797e-06
Log-SPD train max |mean|: 7.187727533164434e-07
Log-SPD train max |std - 1|: 2.1457672119140625e-06

Train spectrum: {'min_eigenvalue_min': 8.714945920473838e-07, 'min_eigenvalue_median': 2.5683925741759595e-06, 'max_eigenvalue_median': 0.0005576540715992451, 'condition_number_median': 224.7659454345703, 'condition_number_p95': 573.766357421875, 'condition_number_max': 2489.1533203125}
Validation spectrum: {'min_eigenvalue_min': 2.3346690340986243e-06, 'min_eigenvalue_median': 9.305320418206975e-06, 'max_eigenvalue_median': 0.0013601714745163918, 'condition_number_median': 182.4652099609375, 'condition_number_p95': 409.4076232910156, 'condition_number_max': 543.7298583984375}


### Feasibility conclusion

The Phase-2A feature gate passes if:

- Covariance and Log-SPD are both 78-dimensional;
- no feature is constant on the training period;
- standardized training means are approximately zero and standard deviations approximately one;
- all regularized covariance eigenvalues are positive;
- condition numbers remain finite and numerically manageable.

Passing this gate establishes numerical validity only. It does **not** establish predictive value.


## 2. Controlled frozen-CDE ablation

Train three otherwise matched models:

- **CDE:** cached 128-dimensional CDE condition only;
- **Covariance:** CDE plus a 78-D ordinary covariance representation;
- **Log-SPD:** CDE plus a 78-D Log-Euclidean SPD representation.

Covariance and Log-SPD therefore have identical auxiliary dimensionality and identical residual-projection capacity. The residual projections are zero-initialized, and the trainer resets RNG after variant-specific module construction so all variants receive the same subsequent stochastic training stream.

The test split remains untouched.


### 2.1 CDE control


In [7]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant cde \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --output-dir /kaggle/working/phase2_cde \
    --device cuda \
    --resume


PHASE 2 FROZEN-CDE SPD ABLATION
Variant: cde
Device: cuda
Input dimension: 128
Extra dimension: 0
Parameters: 826433
Epoch 001/050 | train=0.939270 | val=1.765001
Epoch 002/050 | train=0.823088 | val=1.733040
Epoch 003/050 | train=0.773733 | val=1.667188
Epoch 004/050 | train=0.767575 | val=1.668164
Epoch 005/050 | train=0.737280 | val=1.618074
Epoch 006/050 | train=0.724777 | val=1.612832
Epoch 007/050 | train=0.733843 | val=1.608916
Epoch 008/050 | train=0.710975 | val=1.599657
Epoch 009/050 | train=0.725829 | val=1.587700
Epoch 010/050 | train=0.727528 | val=1.599270
Epoch 011/050 | train=0.694631 | val=1.618166
Epoch 012/050 | train=0.684763 | val=1.609473
Epoch 013/050 | train=0.710739 | val=1.596442
Epoch 014/050 | train=0.683700 | val=1.605483
Epoch 015/050 | train=0.700721 | val=1.623033
Epoch 016/050 | train=0.698409 | val=1.621673
Epoch 017/050 | train=0.683270 | val=1.615558
Epoch 018/050 | train=0.677342 | val=1.621499
Epoch 019/050 | train=0.702325 | val=1.610429
Epoch 020

### 2.2 Ordinary covariance control


In [8]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant covariance \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --output-dir /kaggle/working/phase2_covariance \
    --device cuda \
    --resume


PHASE 2 FROZEN-CDE SPD ABLATION
Variant: covariance
Device: cuda
Input dimension: 206
Extra dimension: 78
Parameters: 836545
Epoch 001/050 | train=0.939176 | val=1.763749
Epoch 002/050 | train=0.822845 | val=1.730846
Epoch 003/050 | train=0.772882 | val=1.664882
Epoch 004/050 | train=0.766305 | val=1.660598
Epoch 005/050 | train=0.735857 | val=1.610577
Epoch 006/050 | train=0.724294 | val=1.608805
Epoch 007/050 | train=0.733053 | val=1.603907
Epoch 008/050 | train=0.709846 | val=1.593439
Epoch 009/050 | train=0.724513 | val=1.583216
Epoch 010/050 | train=0.727388 | val=1.599201
Epoch 011/050 | train=0.694420 | val=1.609874
Epoch 012/050 | train=0.684179 | val=1.597535
Epoch 013/050 | train=0.709615 | val=1.585181
Epoch 014/050 | train=0.683628 | val=1.597427
Epoch 015/050 | train=0.699243 | val=1.612962
Epoch 016/050 | train=0.696932 | val=1.615292
Epoch 017/050 | train=0.683028 | val=1.606966
Epoch 018/050 | train=0.675630 | val=1.611925
Epoch 019/050 | train=0.699192 | val=1.615751
E

### 2.3 Log-Euclidean SPD


In [9]:
!PYTHONUNBUFFERED=1 python -u scripts/train_feature_ablation.py \
    --variant log_spd \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --output-dir /kaggle/working/phase2_log_spd \
    --device cuda \
    --resume


PHASE 2 FROZEN-CDE SPD ABLATION
Variant: log_spd
Device: cuda
Input dimension: 206
Extra dimension: 78
Parameters: 836545
Epoch 001/050 | train=0.939192 | val=1.764549
Epoch 002/050 | train=0.822854 | val=1.731982
Epoch 003/050 | train=0.773314 | val=1.665603
Epoch 004/050 | train=0.766942 | val=1.664665
Epoch 005/050 | train=0.736257 | val=1.614126
Epoch 006/050 | train=0.724047 | val=1.610861
Epoch 007/050 | train=0.732716 | val=1.606132
Epoch 008/050 | train=0.709549 | val=1.598367
Epoch 009/050 | train=0.724585 | val=1.585029
Epoch 010/050 | train=0.726319 | val=1.599534
Epoch 011/050 | train=0.692933 | val=1.614611
Epoch 012/050 | train=0.683725 | val=1.604594
Epoch 013/050 | train=0.709233 | val=1.587741
Epoch 014/050 | train=0.682208 | val=1.607974
Epoch 015/050 | train=0.698900 | val=1.615218
Epoch 016/050 | train=0.696386 | val=1.629472
Epoch 017/050 | train=0.681145 | val=1.604922
Epoch 018/050 | train=0.673450 | val=1.611828
Epoch 019/050 | train=0.697502 | val=1.601474
Epoc

### Training summary


In [10]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

runs = {
    "CDE": Path(
        "/kaggle/working/phase2_cde"
    ),
    "Covariance": Path(
        "/kaggle/working/phase2_covariance"
    ),
    "Log-SPD": Path(
        "/kaggle/working/phase2_log_spd"
    ),
}

rows = []

for name, path in (
    runs.items()
):
    with (
        path / "summary.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        result = json.load(
            handle
        )

    projection_norm = np.nan

    if name != "CDE":
        checkpoint = torch.load(
            path / "best.pt",
            map_location="cpu",
            weights_only=False,
        )

        projection_norm = float(
            checkpoint[
                "model_state_dict"
            ][
                "residual_projection.weight"
            ].norm()
        )

    rows.append(
        {
            "Variant": name,
            "Best epoch": result[
                "best_epoch"
            ],
            "Validation loss": result[
                "best_val_loss"
            ],
            "Extra dim": result[
                "extra_dim"
            ],
            "Parameters": result[
                "parameter_count"
            ],
            "Residual weight norm": (
                projection_norm
            ),
        }
    )

training_results = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "Variant"
    )
)

display(
    training_results
)


,Best epoch,Validation loss,Extra dim,Parameters,Residual weight norm
Variant,,,,,
CDE,9,1.587700,0,826433,NaN
Covariance,9,1.583216,78,836545,1.484339
Log-SPD,9,1.585029,78,836545,1.830731


Training loss is only an intermediate diagnostic. The Phase-2A decision is based primarily on the validation probabilistic metrics below. In particular, Log-SPD must outperform the **ordinary covariance control**, not merely the CDE baseline, to justify a manifold-specific claim.


## 3. Validation-only probabilistic evaluation

Evaluate the best-validation checkpoint from each run using the same protocol:

- 1,256 validation dates;
- 256 generated scenarios per date;
- common sampling seed;
- inverse transformation to raw decimal excess returns;
- CRPS, Energy Score, and interval calibration.

No test observations are evaluated.


### 3.1 CDE control


In [11]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant cde \
    --run-dir /kaggle/working/phase2_cde \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --batch-size 32 \
    --device cuda


PHASE 2 VALIDATION EVALUATION
Variant: cde
Checkpoint epoch: 9
Checkpoint validation loss: 1.5876998479958553
Validation samples: 1256
Scenarios per date: 256

cde: 100%|██████████████████████████████████████| 40/40 [02:18<00:00,  3.47s/it]

CRPS: 0.008557478106706007
Energy Score: 0.0352614411512724

Calibration:
{'level': 0.5, 'picp': 0.33844214437367304, 'ace': -0.16155785562632696}
{'level': 0.8, 'picp': 0.5916932059447984, 'ace': -0.2083067940552017}
{'level': 0.9, 'picp': 0.7045514861995754, 'ace': -0.1954485138004246}
{'level': 0.95, 'picp': 0.7809846072186837, 'ace': -0.16901539278131628}
{'level': 0.99, 'picp': 0.8714835456475584, 'ace': -0.11851645435244162}

TEST SPLIT WAS NOT EVALUATED.


### 3.2 Ordinary covariance control


In [12]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant covariance \
    --run-dir /kaggle/working/phase2_covariance \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --batch-size 32 \
    --device cuda


PHASE 2 VALIDATION EVALUATION
Variant: covariance
Checkpoint epoch: 9
Checkpoint validation loss: 1.583216152373393
Validation samples: 1256
Scenarios per date: 256

covariance: 100%|███████████████████████████████| 40/40 [02:20<00:00,  3.52s/it]

CRPS: 0.008581514872238353
Energy Score: 0.03534398895307809

Calibration:
{'level': 0.5, 'picp': 0.34779723991507433, 'ace': -0.15220276008492567}
{'level': 0.8, 'picp': 0.6014463906581741, 'ace': -0.19855360934182598}
{'level': 0.9, 'picp': 0.7122478768577495, 'ace': -0.18775212314225054}
{'level': 0.95, 'picp': 0.7866242038216561, 'ace': -0.16337579617834386}
{'level': 0.99, 'picp': 0.8748009554140127, 'ace': -0.11519904458598729}

TEST SPLIT WAS NOT EVALUATED.


### 3.3 Log-Euclidean SPD


In [13]:
!PYTHONUNBUFFERED=1 python -u scripts/evaluate_diffusion.py \
    --variant log_spd \
    --run-dir /kaggle/working/phase2_log_spd \
    --cde-run-dir /kaggle/working/phase0_cde \
    --feature-dir /kaggle/working/phase2_features \
    --batch-size 32 \
    --device cuda


PHASE 2 VALIDATION EVALUATION
Variant: log_spd
Checkpoint epoch: 9
Checkpoint validation loss: 1.585029155965064
Validation samples: 1256
Scenarios per date: 256

log_spd: 100%|██████████████████████████████████| 40/40 [02:20<00:00,  3.52s/it]

CRPS: 0.008569843501883462
Energy Score: 0.035303485621606186

Calibration:
{'level': 0.5, 'picp': 0.33996815286624205, 'ace': -0.16003184713375795}
{'level': 0.8, 'picp': 0.5943471337579618, 'ace': -0.20565286624203827}
{'level': 0.9, 'picp': 0.7075371549893843, 'ace': -0.19246284501061572}
{'level': 0.95, 'picp': 0.7847664543524416, 'ace': -0.16523354564755832}
{'level': 0.99, 'picp': 0.8749336518046709, 'ace': -0.11506634819532913}

TEST SPLIT WAS NOT EVALUATED.


### Final comparison


In [14]:
rows = []

for name, path in (
    runs.items()
):
    with (
        path / "summary.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        train_result = json.load(
            handle
        )

    with (
        path
        / "validation_probabilistic_metrics.json"
    ).open(
        encoding="utf-8"
    ) as handle:
        eval_result = json.load(
            handle
        )

    mean_abs_ace = float(
        np.mean(
            [
                abs(
                    item[
                        "ace"
                    ]
                )
                for item in eval_result[
                    "calibration"
                ]
            ]
        )
    )

    rows.append(
        {
            "Variant": name,
            "Best epoch": train_result[
                "best_epoch"
            ],
            "Validation loss": train_result[
                "best_val_loss"
            ],
            "CRPS": eval_result[
                "crps_mean"
            ],
            "Energy Score": eval_result[
                "energy_score"
            ],
            "Mean |ACE|": (
                mean_abs_ace
            ),
        }
    )

phase2_results = (
    pd.DataFrame(
        rows
    )
    .set_index(
        "Variant"
    )
)

display(
    phase2_results
)


,Best epoch,Validation loss,CRPS,Energy Score,Mean |ACE|
Variant,,,,,
CDE,9,1.587700,0.008557,0.035261,0.170569
Covariance,9,1.583216,0.008582,0.035344,0.163417
Log-SPD,9,1.585029,0.008570,0.035303,0.167689


## 4. Phase 2A decision

Interpret the computed comparison table above using the predeclared gate.

The canonical result is **mixed rather than supportive of SPD geometry**:

- ordinary covariance improves validation diffusion loss and calibration, but does not improve CRPS or Energy Score;
- Log-SPD provides only a small validation-loss/calibration improvement over CDE;
- Log-SPD does not outperform the matched ordinary-covariance control on the principal probabilistic forecasting metrics.

Therefore the evidence does not support attributing useful incremental predictive information to the Log-Euclidean SPD representation.

**Decision:** reject the current Log-SPD augmentation. Do not spend compute on multi-seed confirmation, do not tune the eigenvalue floor post hoc, and do not proceed to Bures geometry merely to rescue this branch.

The architecture carried forward remains the validated **Neural CDE-conditioned diffusion** from Phase 0. Ordinary covariance is retained only as an informative scientific control.

The held-out test split remains untouched.


## Backup

Persist the canonical Phase-2A feature cache, controlled ablations, and validation-only evaluation artifacts to the central Kaggle dataset.


In [15]:
import shutil
import kagglehub

HANDLE = "amirentezari/diffusion-portfolio"

try:
    artifact_path = Path(
        kagglehub.dataset_download(
            HANDLE
        )
    )

except Exception as error:
    artifact_path = None

    print(
        "Existing backup unavailable:",
        error,
    )

staging = Path(
    "/kaggle/working/artifact_sync"
)

if staging.exists():
    shutil.rmtree(
        staging
    )

if artifact_path is not None:
    shutil.copytree(
        artifact_path,
        staging,
    )

else:
    staging.mkdir(
        parents=True,
        exist_ok=True,
    )

backups = {
    Path(
        "/kaggle/working/phase2_features"
    ): (
        staging
        / "phase2"
        / "features"
    ),

    Path(
        "/kaggle/working/phase2_cde"
    ): (
        staging
        / "phase2"
        / "cde_control"
    ),

    Path(
        "/kaggle/working/phase2_covariance"
    ): (
        staging
        / "phase2"
        / "covariance_control"
    ),

    Path(
        "/kaggle/working/phase2_log_spd"
    ): (
        staging
        / "phase2"
        / "log_spd"
    ),
}

for source, destination in (
    backups.items()
):
    if not source.exists():
        raise FileNotFoundError(
            source
        )

    if destination.exists():
        shutil.rmtree(
            destination
        )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        source,
        destination,
    )

    print(
        f"Prepared: {source} -> {destination}"
    )

try:
    kagglehub.dataset_upload(
        HANDLE,
        str(
            staging
        ),
        version_notes=(
            "Complete Phase 2A covariance and "
            "Log-Euclidean SPD controlled ablation"
        ),
    )

    print(
        "✅ Complete Phase 2A safely backed up"
    )

except Exception as error:
    print(
        "Backup upload skipped:",
        error,
    )


Prepared: /kaggle/working/phase2_features -> /kaggle/working/artifact_sync/phase2/features
Prepared: /kaggle/working/phase2_cde -> /kaggle/working/artifact_sync/phase2/cde_control
Prepared: /kaggle/working/phase2_covariance -> /kaggle/working/artifact_sync/phase2/covariance_control
Prepared: /kaggle/working/phase2_log_spd -> /kaggle/working/artifact_sync/phase2/log_spd
Uploading Dataset https://kaggle.com/datasets/amirentezari/diffusion-portfolio ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpbx0at7jn/archive.zip


Uploading: 100%|██████████| 341M/341M [00:17<00:00, 20.1MB/s] 

Upload successful: /tmp/tmpbx0at7jn/archive.zip (326MB)


Your dataset version has been created.
Files are being processed...
See at: https://kaggle.com/datasets/amirentezari/diffusion-portfolio
✅ Complete Phase 2A safely backed up
